IMPORTAÇÃO DO ARQUIVO DE LOG

In [2]:
###################################################
### OneDrive sincronizado no computador
### Importar dados de auditoria
###################################################

import pandas as pd
from pathlib import Path

arquivo = Path.home() / (
    r"OneDrive - Instituto Presbiteriano Mackenzie"
    r"\Inovação e Tecnologia Aplicada"
    r"\[Auditoria - FLAN]"
    r"\script.csv"
)

df = pd.read_csv(
    arquivo,
    sep=",",          # ajuste caso o CSV utilize outro separador
    encoding="utf-8"
)

print(df.shape)
print(df.head())

C:\Users\4501050\AppData\Local\Temp\ipykernel_22296\2814581433.py:16: DtypeWarning: Columns (3,15,18,37,38,39,82,102,124,125,133,154,164,171,268) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


(281117, 284)
      AUDITID AUDITACTION    USERNAME LOGINNAME HOSTNAME  \
0  1131401875           U     $SYSTEM       NaN      NaN   
1  1131401875           O     $SYSTEM       NaN      NaN   
2  1131401889           U  1005000357       NaN      NaN   
3  1131401889           O  1005000357       NaN      NaN   
4  1131401899           U  1005000357       NaN      NaN   

                                APPLICATIONMODULE       MACHINEIP      LOGID  \
0            RM.Fin.Parametros.FinBaixaParamsData  192.168.25.167  186746575   
1            RM.Fin.Parametros.FinBaixaParamsData  192.168.25.167  186746576   
2  RM.Fin.Parametros.FinNovoModeloBaixaParamsData      10.0.1.134  186746577   
3  RM.Fin.Parametros.FinNovoModeloBaixaParamsData      10.0.1.134  186746578   
4            RM.Fin.Parametros.FinBaixaParamsData      10.0.1.134  186746579   

   PARENTLOGID   AUDITID.1  ... CPFCNPJLCDPR MODALIDADELCDPR  \
0          NaN  1131401875  ...          NaN             NaN   
1  186746575.0  

In [3]:
###################################################
### Importar inicialmente todas as colunas como texto
###################################################

df = pd.read_csv(
    arquivo,
    sep=None,
    engine="python",
    encoding="utf-8",
    dtype=str
)

AUDITORIA DOS ARQUIVOS

In [4]:
#Auditoria_01 (A) - Primeiro, entenda como os pares estão distribuídos
df["AUDITACTION"].value_counts(dropna=False)

AUDITACTION
U    128394
O    128394
I     18548
D      5781
Name: count, dtype: int64

In [5]:
#Auditoria_01 (B) - Primeiro, entenda como os pares estão distribuídos

df.groupby(["AUDITID", "AUDITACTION"]).size().unstack(fill_value=0).head(20)

AUDITACTION,D,I,O,U
AUDITID,,,,
1131401875,0,0,1,1
1131401889,0,0,1,1
1131401899,0,0,1,1
1131401912,0,0,1,1
1131401921,0,0,1,1
1131401960,0,0,1,1
1131402066,0,1,0,0
1131402075,0,0,1,1
1131402380,0,0,1,1


In [6]:
#Auditoria_02 (A) - Pegue um AUDITID e compare manualmente

audit_id = "1131401875"

teste = df[df["AUDITID"].astype(str) == audit_id]

display(teste)

,"﻿""AUDITID""",AUDITACTION,USERNAME,LOGINNAME,HOSTNAME,APPLICATIONMODULE,MACHINEIP,LOGID,PARENTLOGID,AUDITID,...,CPFCNPJLCDPR,MODALIDADELCDPR,STATUSNEGATIVACAO,STATUSTERCEIRIZACAO,QRCODEPIX,CODCOLCONVENIO,STATUSAPROVACAO,DATACONTROLEAPROVACAO,IDFLUIG,IDNATRENDIMENTOIRRF
0,1131401875,U,$SYSTEM,NaN,NaN,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,NaN,1131401875,...,NaN,NaN,0,0,NaN,1,0,NaN,NaN,NaN
1,1131401875,O,$SYSTEM,NaN,NaN,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746576,186746575,1131401875,...,NaN,NaN,0,0,NaN,1,0,NaN,NaN,NaN


In [7]:
#Auditoria_02 (B) - Pegue um AUDITID e compare manualmente

display(
    teste[
        [
            "AUDITID",
            "AUDITACTION",
            "USERNAME",
            "APPLICATIONMODULE",
            "MACHINEIP",
            "LOGID",
            "PARENTLOGID"
        ]
    ]
)

,AUDITID,AUDITACTION,USERNAME,APPLICATIONMODULE,MACHINEIP,LOGID,PARENTLOGID
0,1131401875,U,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,NaN
1,1131401875,O,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746576,186746575


In [8]:
#Auditoria_03 (A) - Comparar O × U para um AUDITID
import pandas as pd

def comparar_auditid(df, audit_id):

    dados = df[
        df["AUDITID"].astype(str) == str(audit_id)
    ].copy()

    antigo = dados[dados["AUDITACTION"] == "O"]
    novo = dados[dados["AUDITACTION"] == "U"]

    print(f"AUDITID: {audit_id}")
    print(f"Registros O: {len(antigo)}")
    print(f"Registros U: {len(novo)}")

    if len(antigo) != 1 or len(novo) != 1:
        print("Não existe exatamente um par O/U para este AUDITID.")
        return None

    antigo = antigo.iloc[0]
    novo = novo.iloc[0]

    alteracoes = []

    for coluna in df.columns:

        valor_antigo = antigo[coluna]
        valor_novo = novo[coluna]

        # Ambos NaN
        if pd.isna(valor_antigo) and pd.isna(valor_novo):
            continue

        # Normaliza para comparação
        va = "" if pd.isna(valor_antigo) else str(valor_antigo)
        vn = "" if pd.isna(valor_novo) else str(valor_novo)

        if va != vn:

            alteracoes.append({
                "AUDITID": audit_id,
                "CAMPO": coluna,
                "VALOR_ANTERIOR": valor_antigo,
                "VALOR_NOVO": valor_novo
            })

    return pd.DataFrame(alteracoes)

In [9]:
#Auditoria_03 (B) - Comparar O × U para um AUDITID

resultado = comparar_auditid(df, "1131401875")
display(resultado)

AUDITID: 1131401875
Registros O: 1
Registros U: 1


,AUDITID,CAMPO,VALOR_ANTERIOR,VALOR_NOVO
0,1131401875,AUDITACTION,O,U
1,1131401875,LOGID,186746576,186746575
2,1131401875,PARENTLOGID,186746575,NaN
3,1131401875,AUDITACTION.1,O,U
4,1131401875,CNABSTATUS,0,2
5,1131401875,RECMODIFIEDON,08/09/2026 23:58:39,09/09/2026 00:00:17


In [10]:
#Auditoria_04 (A) - Não devemos comparar algumas colunas

colunas_ignorar = [
    "AUDITACTION",
    "LOGID",
    "PARENTLOGID"
]

In [11]:
#Auditoria_04 (B) - Não devemos comparar algumas colunas

def comparar_auditid(df, audit_id):

    colunas_ignorar = {
        "AUDITACTION",
        "LOGID",
        "PARENTLOGID"
    }

    dados = df[
        df["AUDITID"].astype(str) == str(audit_id)
    ].copy()

    antigo = dados[dados["AUDITACTION"] == "O"]
    novo = dados[dados["AUDITACTION"] == "U"]

    if len(antigo) != 1 or len(novo) != 1:
        return None

    antigo = antigo.iloc[0]
    novo = novo.iloc[0]

    alteracoes = []

    for coluna in df.columns:

        if coluna in colunas_ignorar:
            continue

        va = antigo[coluna]
        vn = novo[coluna]

        if pd.isna(va) and pd.isna(vn):
            continue

        va_comp = "" if pd.isna(va) else str(va).strip()
        vn_comp = "" if pd.isna(vn) else str(vn).strip()

        if va_comp != vn_comp:

            alteracoes.append({
                "AUDITID": audit_id,
                "CAMPO": coluna,
                "VALOR_ANTERIOR": va,
                "VALOR_NOVO": vn
            })

    return pd.DataFrame(alteracoes)

Auditoria_05 - Agora vem a parte realmente útil: analisar todos os 281 mil registros

In [12]:
#Auditoria_05 (A) - Agora vem a parte realmente útil: analisar todos os 281 mil registros

alteracoes = []

colunas_ignorar = {
    "AUDITACTION",
    "LOGID",
    "PARENTLOGID"
}

for audit_id, grupo in df.groupby("AUDITID", sort=False):

    antigo = grupo[grupo["AUDITACTION"] == "O"]
    novo = grupo[grupo["AUDITACTION"] == "U"]

    # Só processa pares 1 O / 1 U
    if len(antigo) != 1 or len(novo) != 1:
        continue

    antigo = antigo.iloc[0]
    novo = novo.iloc[0]

    for coluna in df.columns:

        if coluna in colunas_ignorar:
            continue

        va = antigo[coluna]
        vn = novo[coluna]

        if pd.isna(va) and pd.isna(vn):
            continue

        va_comp = "" if pd.isna(va) else str(va).strip()
        vn_comp = "" if pd.isna(vn) else str(vn).strip()

        if va_comp != vn_comp:

            alteracoes.append({
                "AUDITID": audit_id,
                "USERNAME": novo.get("USERNAME"),
                "APPLICATIONMODULE": novo.get("APPLICATIONMODULE"),
                "MACHINEIP": novo.get("MACHINEIP"),
                "LOGID_U": novo.get("LOGID"),
                "LOGID_O": antigo.get("LOGID"),
                "PARENTLOGID": antigo.get("PARENTLOGID"),
                "CAMPO_ALTERADO": coluna,
                "VALOR_ANTERIOR": va,
                "VALOR_NOVO": vn
            })

df_alteracoes = pd.DataFrame(alteracoes)

In [ ]:
#Auditoria_05 (B) - Agora vem a parte realmente útil: analisar todos os 281 mil registros
print(df_alteracoes.shape)

display(df_alteracoes.head(50))

(780307, 10)


,AUDITID,USERNAME,APPLICATIONMODULE,MACHINEIP,LOGID_U,LOGID_O,PARENTLOGID,CAMPO_ALTERADO,VALOR_ANTERIOR,VALOR_NOVO
0,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,AUDITACTION.1,O,U
1,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,CNABSTATUS,0,2
2,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,RECMODIFIEDON,08/09/2026 23:58:39,09/09/2026 00:00:17
3,1131401889,1005000357,RM.Fin.Parametros.FinNovoModeloBaixaParamsData,10.0.1.134                                   ,186746577,186746578,186746577,AUDITACTION.1,O,U
4,1131401889,1005000357,RM.Fin.Parametros.FinNovoModeloBaixaParamsData,10.0.1.134                                   ,186746577,186746578,186746577,RECMODIFIEDON,08/09/2026 23:54:17,09/09/2026 00:00:33
5,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,AUDITACTION.1,O,U
6,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,STATUSLAN,0,4
7,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,DATABAIXA,NaN,09/09/2026 00:00:00
8,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,DATAPAG,NaN,09/09/2026 00:00:00
9,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,VALORBAIXADO,"0,0000","251,8500"


In [14]:
#Auditoria_07 (A) - Identificar quais campos são mais alterados

(
    df_alteracoes["CAMPO_ALTERADO"]
    .value_counts()
    .head(30)
)

CAMPO_ALTERADO
AUDITACTION.1            127675
RECMODIFIEDON            107914
RECMODIFIEDBY             54330
STATUSLAN                 42689
CNABSTATUS                41410
USUARIO                   38181
DATAALTERACAO             37399
DATABAIXA                 35930
VALORORIGINALBX           35480
DATAPAG                   35356
VALOROPERACAODESCONTO     35347
VALORBAIXADO              34109
CNABCODRETORNO            28713
CNABNOSSONUMERO           12552
IDBOLETO                  12481
CNABCOMANDO               12125
CODIGOBARRA               10027
IPTE                      10027
IDMOV                      7529
HISTORICO                  6898
DATACANCELAMENTO           6755
NUMERODOCUMENTO            5681
VALORDESCONTOBX            4026
VALORAUXILIAR              3889
VALORJUROSBX               2982
VALORMULTABX               2826
VALORCAPBX                 2709
CODCXA                     1813
IDBORDERO                  1812
CNABBANCO                  1779
Name: count, dtype: int64

In [15]:
#Auditoria_07 (B) - Identificar quais campos são mais alterados

display(
    df_alteracoes["CAMPO_ALTERADO"]
    .value_counts()
    .rename_axis("CAMPO")
    .reset_index(name="QUANTIDADE")
)

,CAMPO,QUANTIDADE
0,AUDITACTION.1,127675
1,RECMODIFIEDON,107914
2,RECMODIFIEDBY,54330
3,STATUSLAN,42689
4,CNABSTATUS,41410
...,...,...
95,VALOROP3,2
96,SERIEDOCUMENTO,1
97,CODUSUDESBLOQUEIO1,1
98,CODFILIAL,1


In [16]:
#Auditoria_08 (A) - Quais usuários realizaram mais alterações

display(
    df_alteracoes["USERNAME"]
    .value_counts()
    .rename_axis("USERNAME")
    .reset_index(name="QUANTIDADE")
    .head(30)
)


,USERNAME,QUANTIDADE
0,AMS0586,441781
1,rubeus,37731
2,1132769,33561
3,AMS0585,27816
4,6014355,19570
5,RM,14752
6,$SYSTEM,14717
7,1145662,12323
8,1001000285,11179
9,1142107,10486


In [17]:
#Auditoria_08 (B) - Quais usuários realizaram mais alterações

display(
    df_alteracoes.groupby("USERNAME")["AUDITID"]
    .nunique()
    .sort_values(ascending=False)
    .rename("AUDITORIAS")
    .reset_index()
)

,USERNAME,AUDITORIAS
0,AMS0586,59027
1,1132769,6761
2,rubeus,5868
3,AMS0585,5066
4,$SYSTEM,4468
...,...,...
597,10331583,1
598,10308846,1
599,41093747153,1
600,53591356891,1


In [ ]:
#Auditoria_09 (A) - Há uma validação que considero fundamental
#Antes de processarmos definitivamente o arquivo inteiro, rode

diagnostico = (
    df.groupby("AUDITID")["AUDITACTION"]
      .agg(
          TOTAL="size",
          ACOES=lambda x: ",".join(sorted(x.dropna().astype(str).unique()))
      )
)

display(diagnostico["ACOES"].value_counts())


ACOES
O,U    127777
I       18548
D        5781
Name: count, dtype: int64

In [19]:
#Auditoria_09 (B) - Há uma validação que considero fundamental
#Antes de processarmos definitivamente o arquivo inteiro, rode

contagem = (
    df.groupby(["AUDITID", "AUDITACTION"])
      .size()
      .unstack(fill_value=0)
)

display(contagem.head(30))

AUDITACTION,D,I,O,U
AUDITID,,,,
1131401875,0,0,1,1
1131401889,0,0,1,1
1131401899,0,0,1,1
1131401912,0,0,1,1
1131401921,0,0,1,1
1131401960,0,0,1,1
1131402066,0,1,0,0
1131402075,0,0,1,1
1131402380,0,0,1,1


In [ ]:
#Auditoria_10 (A) - Verifique se há pares problemáticos

contagem = (
    df.groupby(["AUDITID", "AUDITACTION"])
      .size()
      .unstack(fill_value=0)
)

# Garante as colunas mesmo que alguma ação não exista
for col in ["O", "U"]:
    if col not in contagem.columns:
        contagem[col] = 0

contagem["PAR_VALIDO"] = (
    (contagem["O"] == 1) &
    (contagem["U"] == 1)
)

print("Resumo:")
print(contagem["PAR_VALIDO"].value_counts())

print("\nDistribuição O/U:")
print(
    contagem.groupby(["O", "U"])
            .size()
            .sort_values(ascending=False)
            .head(20)
)

Resumo:
PAR_VALIDO
True     127675
False     24431
Name: count, dtype: int64

Distribuição O/U:
O   U 
1   1     127675
0   0      24329
5   5         23
2   2         20
4   4         20
3   3         11
7   7          4
6   6          3
12  12         3
28  28         3
24  24         3
17  17         3
13  13         2
11  11         1
8   8          1
10  10         1
22  22         1
18  18         1
30  30         1
37  37         1
dtype: int64


In [21]:
#Auditoria_10 (B) - Se existirem muitos AUDITID com múltiplos O ou U, não descarte esses registros. Guarde-os em uma tabela de exceções

auditids_excecao = contagem[
    ~contagem["PAR_VALIDO"]
].reset_index()

display(auditids_excecao.head(50))


AUDITACTION,AUDITID,D,I,O,U,PAR_VALIDO
0,1131402066,0,1,0,0,False
1,1131402396,0,1,0,0,False
2,1131402475,0,1,0,0,False
3,1131402537,0,1,0,0,False
4,1131402612,0,1,0,0,False
5,1131402789,0,1,0,0,False
6,1131402927,0,1,0,0,False
7,1131402954,0,1,0,0,False
8,1131403248,0,1,0,0,False
9,1131408911,0,1,0,0,False


In [22]:
#Auditoria_11 - Crie uma base somente com os pares válidos

ids_validos = contagem[
    contagem["PAR_VALIDO"]
].index

df_validos = df[
    df["AUDITID"].isin(ids_validos)
].copy()

print("Registros para comparação:", len(df_validos))


Registros para comparação: 255350


#Auditoria_12 - Gere a base definitiva de alterações

In [23]:
#Auditoria_12 - Gere a base definitiva de alterações

import pandas as pd

colunas_ignorar = {
    "AUDITACTION",
    "LOGID",
    "PARENTLOGID"
}

alteracoes = []

for audit_id, grupo in df_validos.groupby("AUDITID", sort=False):

    registro_o = grupo[
        grupo["AUDITACTION"] == "O"
    ].iloc[0]

    registro_u = grupo[
        grupo["AUDITACTION"] == "U"
    ].iloc[0]

    for coluna in df.columns:

        if coluna in colunas_ignorar:
            continue

        anterior = registro_o[coluna]
        novo = registro_u[coluna]

        # Ambos vazios
        if pd.isna(anterior) and pd.isna(novo):
            continue

        anterior_comp = (
            ""
            if pd.isna(anterior)
            else str(anterior).strip()
        )

        novo_comp = (
            ""
            if pd.isna(novo)
            else str(novo).strip()
        )

        if anterior_comp != novo_comp:

            alteracoes.append({

                "AUDITID": audit_id,

                "USERNAME":
                    registro_u.get("USERNAME"),

                "APPLICATIONMODULE":
                    registro_u.get("APPLICATIONMODULE"),

                "MACHINEIP":
                    registro_u.get("MACHINEIP"),

                "LOGID_U":
                    registro_u.get("LOGID"),

                "LOGID_O":
                    registro_o.get("LOGID"),

                "PARENTLOGID":
                    registro_o.get("PARENTLOGID"),

                "CAMPO_ALTERADO":
                    coluna,

                "VALOR_ANTERIOR":
                    anterior,

                "VALOR_NOVO":
                    novo
            })


df_alteracoes = pd.DataFrame(alteracoes)

print("Alterações identificadas:", len(df_alteracoes))
print("AUDITIDs envolvidos:", df_alteracoes["AUDITID"].nunique())

display(df_alteracoes.head(50))

Alterações identificadas: 780307
AUDITIDs envolvidos: 127675


,AUDITID,USERNAME,APPLICATIONMODULE,MACHINEIP,LOGID_U,LOGID_O,PARENTLOGID,CAMPO_ALTERADO,VALOR_ANTERIOR,VALOR_NOVO
0,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,AUDITACTION.1,O,U
1,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,CNABSTATUS,0,2
2,1131401875,$SYSTEM,RM.Fin.Parametros.FinBaixaParamsData,192.168.25.167                               ,186746575,186746576,186746575,RECMODIFIEDON,08/09/2026 23:58:39,09/09/2026 00:00:17
3,1131401889,1005000357,RM.Fin.Parametros.FinNovoModeloBaixaParamsData,10.0.1.134                                   ,186746577,186746578,186746577,AUDITACTION.1,O,U
4,1131401889,1005000357,RM.Fin.Parametros.FinNovoModeloBaixaParamsData,10.0.1.134                                   ,186746577,186746578,186746577,RECMODIFIEDON,08/09/2026 23:54:17,09/09/2026 00:00:33
5,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,AUDITACTION.1,O,U
6,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,STATUSLAN,0,4
7,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,DATABAIXA,NaN,09/09/2026 00:00:00
8,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,DATAPAG,NaN,09/09/2026 00:00:00
9,1131401899,1005000357,RM.Fin.Parametros.FinBaixaParamsData,10.0.1.134                                   ,186746579,186746580,186746579,VALORBAIXADO,"0,0000","251,8500"


In [24]:
#Auditoria_13 - Classifique o tipo da mudança

def tipo_alteracao(row):

    anterior = row["VALOR_ANTERIOR"]
    novo = row["VALOR_NOVO"]

    anterior_vazio = (
        pd.isna(anterior) or
        str(anterior).strip() == ""
    )

    novo_vazio = (
        pd.isna(novo) or
        str(novo).strip() == ""
    )

    if anterior_vazio and not novo_vazio:
        return "PREENCHIMENTO"

    if not anterior_vazio and novo_vazio:
        return "REMOCAO"

    return "ALTERACAO"


df_alteracoes["TIPO_ALTERACAO"] = (
    df_alteracoes.apply(tipo_alteracao, axis=1)
)

#Auditoria_14 - Gere as tabelas analíticas

In [ ]:
#Auditoria_14 (A) - Gere as tabelas analíticas (Campos mais modificados)

campos = (
    df_alteracoes
    .groupby("CAMPO_ALTERADO")
    .agg(
        ALTERACOES=("CAMPO_ALTERADO", "size"),
        AUDITIDS=("AUDITID", "nunique")
    )
    .reset_index()
    .sort_values(
        "ALTERACOES",
        ascending=False
    )
)

display(campos.head(30))



,CAMPO_ALTERADO,ALTERACOES,AUDITIDS
1,AUDITACTION.1,127675,127675
59,RECMODIFIEDON,107914,107914
58,RECMODIFIEDBY,54330,54330
62,STATUSLAN,42689,42689
13,CNABSTATUS,41410,41410
66,USUARIO,38181,38181
30,DATAALTERACAO,37399,37399
31,DATABAIXA,35930,35930
95,VALORORIGINALBX,35480,35480
36,DATAPAG,35356,35356


In [ ]:
#Auditoria_14 (B) - Gere as tabelas analíticas (Operações por usuário)
#Nesse caso, sugiro contar AUDITID únicos, e não simplesmente linhas


usuarios = (
    df_alteracoes
    .groupby("USERNAME")
    .agg(
        OPERACOES=("AUDITID", "nunique"),
        CAMPOS_ALTERADOS=("CAMPO_ALTERADO", "size")
    )
    .reset_index()
    .sort_values(
        "OPERACOES",
        ascending=False
    )
)

display(usuarios.head(30))

,USERNAME,OPERACOES,CAMPOS_ALTERADOS
595,AMS0586,59027,441781
370,1132769,6761,33561
600,rubeus,5868,37731
594,AMS0585,5066,27816
0,$SYSTEM,4468,14717
568,6014355,3732,19570
596,RM,2240,14752
398,1145662,1968,12323
351,1108603,1740,7926
34,1001000495,1411,6755


In [27]:
#Auditoria_14 (C) - Alterações por módulo

modulos = (
    df_alteracoes
    .groupby("APPLICATIONMODULE")
    .agg(
        OPERACOES=("AUDITID", "nunique"),
        CAMPOS_ALTERADOS=("CAMPO_ALTERADO", "size")
    )
    .reset_index()
    .sort_values(
        "OPERACOES",
        ascending=False
    )
)

display(modulos.head(30))

,APPLICATIONMODULE,OPERACOES,CAMPOS_ALTERADOS
9,RM.Fin.Parametros.FinBaixaParamsData,45459,397546
1,RM.Con.TotvsMessage.ConTotvsMessageServer,26209,128823
13,RM.Fin.Parametros.FinInternoParamsData,24061,103170
12,RM.Fin.Parametros.FinGeraisParamsData,18934,96030
8,RM.Fin.Parametros.FinAprovacaoLancamentoPagarP...,5293,23983
17,RM.Lib.Server.RMSFluigServer,1562,6091
0,.Net SqlClient Data Provider,1356,6780
15,RM.Fin.Parametros.FinNovoModeloBaixaParamsData,1268,3686
6,RM.Fin.Boleto.FinBoletoServer,1086,4280
19,RM.Mov.Parametros.Data.MovGeralRepresParamsData,956,2522


In [28]:
#Auditoria_15 - Salve o resultado

from pathlib import Path

saida = (
    Path.home()
    / "OneDrive - Instituto Presbiteriano Mackenzie"
    / "Inovação e Tecnologia Aplicada"
    / "[Auditoria - FLAN]"
)

df_alteracoes.to_csv(
    saida / "alteracoes_auditoria.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

auditids_excecao.to_csv(
    saida / "excecoes_auditoria.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gravados com sucesso.")

Arquivos gravados com sucesso.
